# Lab 3: Communities

A **community** is a group of nodes with many more links among themselves than to the rest of the network. In a payment network, communities are ecosystems: a supply chain, a regional cluster, a group of firms that trade with each other.

**You will:**
1. watch community detection predict a real-world split (the karate club)
2. learn what modularity measures
3. find communities in the payment network, with a resolution slider
4. see why communities are ecosystems, not industry codes
5. compare two half-years: the part of the story no account-level view can tell

In [ ]:
# Setup
import numpy as np, pandas as pd, networkx as nx
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
import labkit as lk

nodes, edges = lk.load()
name, sector, region = (dict(zip(nodes.node_id, nodes[c])) for c in ("name", "sector", "region"))
HUBS = lk.find_hubs(edges)
full = lk.payment_graph(edges, directed=False, drop=HUBS)
G = full.subgraph([n for n in full if n.startswith("B")]).copy()       # businesses, hubs removed, one year
display(f"Business network without hubs: {G.number_of_nodes():,} businesses, {G.number_of_edges():,} relationships")

## 1. The karate club (Zachary, 1977)

A university karate club of 34 members split in two after a dispute between the instructor and the club officer. Wayne Zachary had recorded who interacted with whom *before* the split. Can the network predict who went where?

Girvan and Newman's method (2002) removes the edge with the highest betweenness, again and again, until the network falls apart. Its first split is our prediction.

In [ ]:
# Predict the split from the friendship network alone
K = nx.karate_club_graph()
real = {n: K.nodes[n]["club"] for n in K}                      # 'Mr. Hi' or 'Officer'
gn = next(nx.community.girvan_newman(K))                       # first split: two groups
gn_lab = {n: i for i, c in enumerate(gn) for n in c}
lv = nx.community.louvain_communities(K, seed=1)
lv_lab = {n: i for i, c in enumerate(lv) for n in c}

# which Girvan-Newman group matches 'Mr. Hi'? count the members placed correctly
hits = max(sum((real[n] == "Mr. Hi") == (gn_lab[n] == g) for n in K) for g in (0, 1))
kp = nx.spring_layout(K, seed=4)
fig = make_subplots(rows=1, cols=3, horizontal_spacing=0.02,
                    subplot_titles=("What actually happened", f"Girvan-Newman: {hits} of 34 right",
                                    f"Louvain: {len(lv)} communities"))
for col, lab in ((1, {n: int(real[n] == "Officer") for n in K}), (2, gn_lab), (3, lv_lab)):
    xs, ys = [], []
    for u, v in K.edges():
        xs += [kp[u][0], kp[v][0], None]
        ys += [kp[u][1], kp[v][1], None]
    fig.add_trace(go.Scatter(x=xs, y=ys, mode="lines", line=dict(width=0.8, color=lk.EDGE), hoverinfo="skip",
                             showlegend=False), 1, col)
    fig.add_trace(go.Scatter(x=[kp[n][0] for n in K], y=[kp[n][1] for n in K], mode="markers+text",
                             text=[str(n) for n in K], textfont=dict(size=8, color="white"), showlegend=False,
                             marker=dict(size=17, color=[lk.PALETTE[lab[n] % 8] for n in K])), 1, col)
fig.update_xaxes(visible=False)
fig.update_yaxes(visible=False)
lk.style(fig, "The network knew who would leave with whom", height=430)

## 2. Modularity: how good is a split?

Modularity Q = (share of edges that fall inside communities) minus (the share you would expect if edges were rewired at random, keeping every node's degree).

* Q near 0: the "communities" are no better than chance
* Q above about 0.3: real structure
* Louvain (2008) and Leiden (2019) are fast algorithms that search for a split with high Q

In [ ]:
# Modularity of three splits of the karate club
rng = np.random.default_rng(0)
rand = {n: rng.integers(0, 2) for n in K}
splits = {"random halves": [{n for n in K if rand[n] == g} for g in (0, 1)],
          "the real split": [{n for n in K if real[n] == c} for c in ("Mr. Hi", "Officer")],
          "Louvain": lv}
q = pd.Series({k: nx.community.modularity(K, v) for k, v in splits.items()})
lk.bar(list(q.index), q.values.round(3), "Modularity: random split vs the real split vs Louvain", ylabel="Q",
       text=[f"{v:.2f}" for v in q.values], height=380)

## 3. Communities in the payment network

In [ ]:
# Louvain on the business network (weight = log of dollars, so a few huge flows don't decide everything)
comms = nx.community.louvain_communities(G, weight="weight", resolution=1.0, seed=1)
comms = sorted(comms, key=len, reverse=True)
Q = nx.community.modularity(G, comms, weight="weight")
lk.bar([f"C{i + 1}" for i in range(len(comms))], [len(c) for c in comms],
       f"{len(comms)} communities, modularity {Q:.2f}", ylabel="businesses", height=380)

In [ ]:
# The same adjacency matrix, two orderings: a random order, then grouped by community
order_c = [n for c in comms for n in sorted(c)]
order_r = list(np.random.default_rng(2).permutation(order_c))
fig, axes = plt.subplots(1, 2, figsize=(13, 6.3), facecolor=lk.SURFACE)
for ax, order, title in ((axes[0], order_r, "Random order"), (axes[1], order_c, "Ordered by community")):
    A = nx.to_scipy_sparse_array(G, nodelist=order, weight=None)
    ax.spy(A, markersize=0.4, color=lk.PALETTE[0])
    ax.set_title(title, fontsize=14)
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("Each dot is a payment relationship: communities appear as dense blocks", fontsize=15, x=0.01, ha="left")
plt.tight_layout()
plt.show()

### The resolution knob

Every community algorithm has a resolution: low values merge communities, high values split them. There is no single right answer. Drag the slider.

In [ ]:
# Resolution slider (communities precomputed at several resolutions; layout fixed)
pos = nx.spring_layout(G, seed=3, iterations=50)
RES = [0.3, 0.6, 1.0, 1.5, 2.0, 3.0, 5.0]
parts = {}
for r in RES:
    cs = sorted(nx.community.louvain_communities(G, weight="weight", resolution=r, seed=1), key=len, reverse=True)
    parts[r] = ({n: i for i, c in enumerate(cs) for n in c}, len(cs), nx.community.modularity(G, cs, weight="weight"))

order = list(G)
def colours(r):
    lab = parts[r][0]
    return [lk.PALETTE[lab[n]] if lab[n] < len(lk.PALETTE) else lk.OTHER for n in order]

xs, ys = [], []
for u, v in G.edges():
    xs += [pos[u][0], pos[v][0], None]
    ys += [pos[u][1], pos[v][1], None]
fig = go.Figure([go.Scatter(x=xs, y=ys, mode="lines", line=dict(width=0.25, color="#dddcd6"), hoverinfo="skip"),
                 go.Scatter(x=[pos[n][0] for n in order], y=[pos[n][1] for n in order], mode="markers",
                            marker=dict(size=5, color=colours(1.0)),
                            hovertext=[f"{name[n]}<br>{sector[n]}, {region[n]}" for n in order], hoverinfo="text")])
steps = [dict(method="restyle", label=f"{r:g}", args=[{"marker.color": [colours(r)]}, [1]]) for r in RES]
fig.update_layout(sliders=[dict(active=RES.index(1.0), steps=steps, y=-0.02,
                                currentvalue=dict(prefix="resolution "))], showlegend=False)
info = pd.DataFrame({"resolution": RES, "communities": [parts[r][1] for r in RES],
                     "modularity": [round(parts[r][2], 3) for r in RES]})
display(info, lk._layout(fig, "Communities at each resolution (largest eight coloured, the rest grey)", 720, legend=False))

In [ ]:
# How close do we get to the planted ecosystems? (In real life there is no answer key.)
_, _, truth = lk.load(truth=True)
eco = truth.set_index("node_id").planted_ecosystem
known = [n for n in G if eco[n].startswith("E")]
score = pd.DataFrame({"resolution": RES,
                      "ARI": [adjusted_rand_score([eco[n] for n in known], [parts[r][0][n] for n in known]) for r in RES],
                      "NMI": [normalized_mutual_info_score([eco[n] for n in known], [parts[r][0][n] for n in known]) for r in RES]})
lp = list(nx.community.label_propagation_communities(G))
lp_lab = {n: i for i, c in enumerate(lp) for n in c}
fig = go.Figure([go.Scatter(x=score.resolution, y=score[c], mode="lines+markers", name=c,
                            line=dict(width=2, color=lk.PALETTE[i])) for i, c in enumerate(["ARI", "NMI"])])
fig.update_xaxes(type="log", title="resolution")
fig.update_yaxes(range=[0, 1], title="agreement with the planted ecosystems")
display(lk.style(fig, "Agreement with the answer key peaks around resolution 2", height=380),
        f"Label propagation (no resolution, very fast): {len(lp)} communities, "
        f"ARI {adjusted_rand_score([eco[n] for n in known], [lp_lab[n] for n in known]):.2f}")

Without an answer key, judge a split by: modularity, stability (rerun with another seed: do the same communities come back?) and whether the communities make business sense.

## 4. Ecosystems, not industry codes

In [ ]:
# Profile the communities at resolution 2
lab2 = parts[2.0][0]
prof = pd.DataFrame({"community": [lab2[n] for n in G], "sector": [sector[n] for n in G],
                     "region": [region[n] for n in G]}, index=list(G))
def top_share(s):
    vc = s.value_counts(normalize=True)
    return pd.Series({"top": vc.index[0], "share": vc.iloc[0]})
p = prof.groupby("community").agg(members=("sector", "size"))
p = p.join(prof.groupby("community").sector.apply(top_share).unstack().rename(columns={"top": "main sector", "share": "sector purity"}))
p = p.join(prof.groupby("community").region.apply(top_share).unstack().rename(columns={"top": "main region", "share": "region purity"}))
p["label"] = [f"{r} · {s}" for r, s in zip(p["main region"], p["main sector"])]
p = p.sort_values("members", ascending=False)
p[p.members >= 15].round(2)

Regions are much purer than sectors. A community is a **place and a supply chain**: hospitals with their suppliers and cleaners, carmakers with their parts makers and truckers. An industry code alone would never group them.

In [ ]:
# Community-level map: one bubble per community, links = dollars between communities
amount = nx.get_edge_attributes(G, "amount")
flows = {}
for (u, v), a in amount.items():
    cu, cv = lab2[u], lab2[v]
    if cu != cv:
        k = (min(cu, cv), max(cu, cv))
        flows[k] = flows.get(k, 0) + a
big = list(p[p.members >= 15].index)
QG = nx.Graph()
QG.add_nodes_from(big)
lw = {k: np.log1p(v) for k, v in flows.items() if k[0] in big and k[1] in big}
for (a, b), w in sorted(lw.items(), key=lambda kv: -kv[1])[:3 * len(big)]:
    QG.add_edge(a, b, weight=w)
qp = nx.spring_layout(QG, seed=2, weight="weight", k=1.4)
cm = lk.colour_map(p.loc[big, "main sector"])
wmax = max(nx.get_edge_attributes(QG, "weight").values())
traces = []
for a, b, d in QG.edges(data=True):
    traces.append(go.Scatter(x=[qp[a][0], qp[b][0]], y=[qp[a][1], qp[b][1]], mode="lines", hoverinfo="skip",
                             showlegend=False, line=dict(width=1 + 7 * (d["weight"] / wmax) ** 4, color=lk.EDGE)))
for s, c in list(cm.items()) + [("Other sectors", lk.OTHER)]:
    mem = [b for b in big if (p.loc[b, "main sector"] == s) or (s == "Other sectors" and p.loc[b, "main sector"] not in cm)]
    if mem:
        traces.append(go.Scatter(x=[qp[b][0] for b in mem], y=[qp[b][1] for b in mem], mode="markers+text", name=s,
                                 text=[p.loc[b, "main region"] for b in mem], textposition="bottom center",
                                 marker=dict(size=[14 + 40 * np.sqrt(p.loc[b, "members"] / p.members.max()) for b in mem],
                                             color=c, line=dict(width=2, color=lk.SURFACE)),
                                 hovertext=[f"{p.loc[b, 'label']}<br>{p.loc[b, 'members']} members<br>sector purity "
                                            f"{p.loc[b, 'sector purity']:.0%}" for b in mem], hoverinfo="text"))
lk._layout(go.Figure(traces), "The network at community level (colour = main sector, label = main region)", 620)

## 5. Communities over time: what no account-level view can see

Run community detection separately on each half-year and match the communities by shared members. Three kinds of event show up:

* **birth**: a new ecosystem forms (firms that used to trade elsewhere start trading with each other)
* **merge**: two ecosystems become one (a new contract wires them together)
* **shrink**: an ecosystem loses members and flow

None of these is a property of a single customer.

In [ ]:
# Detect each half-year separately and match communities by member overlap (Jaccard)
def detect(months, r=1.5):
    g = lk.payment_graph(edges, months, directed=False, drop=HUBS)
    g = g.subgraph([n for n in g if n.startswith("B")]).copy()
    cs = [c for c in sorted(nx.community.louvain_communities(g, weight="weight", resolution=r, seed=1), key=len, reverse=True)
          if len(c) >= 30]
    return g, cs

g1, c1 = detect(lk.MONTHS[:6])
g2, c2 = detect(lk.MONTHS[6:])
label = lambda c: f"{pd.Series([region[n] for n in c]).mode()[0]} · {pd.Series([sector[n] for n in c]).mode()[0]}"
jac = pd.DataFrame([[len(a & b) / len(a | b) for b in c2] for a in c1])

last2 = edges[edges.month.isin(lk.MONTHS[-2:]) & ~edges.src.isin(HUBS) & ~edges.dst.isin(HUBS)]
active2 = set(last2.src) | set(last2.dst)           # businesses still trading in the last two months
born = set()
events = []
for j, b in enumerate(c2):
    parents = [i for i, a in enumerate(c1) if len(a & b) >= 0.25 * len(a)]
    if not parents:
        born.add(j)
        events.append((f"H2 {label(b)}", "birth", len(b), "no H1 community gave it a quarter of its members"))
    elif len(parents) >= 2:
        events.append((f"H2 {label(b)}", "merge", len(b), " + ".join(f"{label(c1[i])} ({len(c1[i])})" for i in parents)))
for i, a in enumerate(c1):
    gone = len(a - active2) / len(a)
    moved = max((len(a & c2[j]) / len(a) for j in born), default=0)
    if gone >= 0.25:
        events.append((f"H1 {label(a)}", "shrink", len(a), f"{gone:.0%} of its members stopped trading by the end of H2"))
    if moved >= 0.10:
        events.append((f"H1 {label(a)}", "feeds the new community", len(a), f"{moved:.0%} of its members moved to it"))
ev = pd.DataFrame(events, columns=["community in H2", "event", "members", "detail"])

# Sankey: where did each H1 community's members go?
src_l = [f"H1 {label(a)}" for a in c1]
dst_l = [f"H2 {label(b)}" for b in c2]
s, t, v = [], [], []
for i, a in enumerate(c1):
    for j, b in enumerate(c2):
        k = len(a & b)
        if k >= 10:
            s.append(i); t.append(len(c1) + j); v.append(k)
    gone = len(a - set().union(*c2))
    if gone >= 10:
        s.append(i); t.append(len(c1) + len(c2)); v.append(gone)
col1 = [lk.PALETTE[i % 8] for i in range(len(c1))]
col2 = ["#0b0b0b" if j in born else col1[max(range(len(c1)), key=lambda i: len(c1[i] & b))] for j, b in enumerate(c2)]
rgba = lambda h, a: f"rgba({int(h[1:3], 16)},{int(h[3:5], 16)},{int(h[5:7], 16)},{a})"
fig = go.Figure(go.Sankey(node=dict(label=src_l + dst_l + ["no relationships left, or a small group"], pad=12,
                                    thickness=14, color=col1 + col2 + [lk.OTHER]),
                          link=dict(source=s, target=t, value=v, color=[rgba(col1[x], 0.35) for x in s])))
display(ev, lk.style(fig, "Members of each H1 community (left) and where they are in H2 (right)", height=700))

In [ ]:
# The account view vs the ecosystem view, for the community that lost the most members
gone_share = [len(a - active2) / len(a) for a in c1]
a = c1[int(np.argmax(gone_share))]
def vol(months, members):
    e = edges[edges.month.isin(months) & (edges.src.isin(members) | edges.dst.isin(members))]
    return pd.concat([e.groupby("src").amount.sum(), e.groupby("dst").amount.sum()]).groupby(level=0).sum()
v1, v2 = vol(lk.MONTHS[:6], a), vol(lk.MONTHS[6:], a)
stay = [n for n in a if n in active2]              # members still trading at the end
own = pd.Series({n: v2.get(n, 0) / v1[n] - 1 for n in stay if v1.get(n, 0) > 0})
inside = lambda g: sum(d["amount"] for _, _, d in g.subgraph(a).edges(data=True))
eco_change = inside(g2) / inside(g1) - 1
fig = go.Figure(go.Histogram(x=own.clip(-1, 1), nbinsx=40, marker=dict(color=lk.PALETTE[0])))
right_first = own.median() > eco_change
fig.add_vline(x=own.median(), line=dict(color=lk.INK, dash="dot"), annotation_text=f"typical member {own.median():+.0%}",
              annotation_position="top right" if right_first else "top left")
fig.add_vline(x=eco_change, line=dict(color=lk.PALETTE[1], width=3),
              annotation_text=f"the ecosystem's internal dollars {eco_change:+.0%}",
              annotation_position="top left" if right_first else "top right")
fig.update_layout(xaxis_title="each still-active member's own volume change, H1 to H2 (capped at +100%)", yaxis_title="members",
                  xaxis_tickformat=".0%", showlegend=False)
lk.style(fig, f"{label(a)}: a typical member is down {-own.median():.0%}; the ecosystem lost "
              f"{max(gone_share):.0%} of its members and {-eco_change:.0%} of its internal dollars", 430)

## Takeaways

* Community detection finds groups that trade with each other; modularity says whether the split beats chance.
* Resolution is a choice: check stability and business sense.
* In payment networks, communities are ecosystems (place plus supply chain), not industry codes.
* Tracked over time, communities show births, merges and shrinking ecosystems: signals that exist only at the network level.